# Дообучение YOLO11 на SportsMOT

Ноутбук подготавливает разметку SportsMOT в формате YOLO и запускает дообучение детектора Ultralytics YOLO11.

Основной сценарий рассчитан на Kaggle Notebook: датасет подключается через Kaggle Input, а рабочие файлы и результаты обучения сохраняются в `/kaggle/working`.

## Что делает ноутбук

1. Устанавливает фиксированную версию Ultralytics.
2. Находит SportsMOT в Kaggle Input.
3. Преобразует `gt/gt.txt` из MOTChallenge-формата в YOLO-разметку.
4. Создает `sportsmot.yaml` для обучения.
5. Запускает дообучение выбранной модели YOLO11.

Для обучения `yolo11n`, `yolo11s` и `yolo11m` ноутбук запускался несколько раз с разным значением `MODEL_NAME`.

## 1. Установка зависимостей

Версия `ultralytics` фиксируется, чтобы обучение было ближе к окружению основного проекта.

In [ ]:
!python -m pip install -q ultralytics==8.4.19 pyyaml tqdm

## 2. Параметры запуска

Если Kaggle смонтировал датасет под другим именем, измените `SPORTSMOT_ROOT` вручную после выполнения ячейки автообнаружения.

In [ ]:
import configparser
import os
import shutil
from collections import defaultdict
from pathlib import Path

import yaml
from tqdm.auto import tqdm


def find_sportsmot_root(input_root: Path = Path("/kaggle/input")) -> Path:
    candidates = [input_root / "sportsmot"]
    if input_root.exists():
        candidates.extend(path for path in input_root.glob("*") if path.is_dir())
        candidates.extend(path for path in input_root.glob("*/*") if path.is_dir())

    for candidate in candidates:
        if (candidate / "dataset" / "train").exists() and (candidate / "dataset" / "val").exists():
            return candidate
    return candidates[0]


SPORTSMOT_ROOT = find_sportsmot_root()
WORKDIR = Path("/kaggle/working")
YOLO_ROOT = WORKDIR / "sportsmot_yolo"

# Для обучения других вариантов поменяйте: yolo11n.pt, yolo11s.pt, yolo11m.pt.
MODEL_NAME = "yolo11s.pt"
RUN_NAME = f"{Path(MODEL_NAME).stem}_sportsmot_finetune"

EPOCHS = 100
IMGSZ = 640
BATCH = 16
DEVICE = 0
WORKERS = 4
SEED = 42

CLASS_ID = 0
CLASS_NAME = "player"
IMAGE_MODE = "symlink"  # symlink быстрее и экономит место; при ошибке будет copy.

print("SPORTSMOT_ROOT =", SPORTSMOT_ROOT)
print("YOLO_ROOT      =", YOLO_ROOT)
print("MODEL_NAME     =", MODEL_NAME)
print("RUN_NAME       =", RUN_NAME)

## 3. Проверка структуры SportsMOT

Ожидается структура `dataset/train/<sequence>/img1` и `dataset/val/<sequence>/gt/gt.txt`.

In [ ]:
train_dir = SPORTSMOT_ROOT / "dataset" / "train"
val_dir = SPORTSMOT_ROOT / "dataset" / "val"

if not train_dir.exists():
    raise FileNotFoundError(f"Не найден train split: {train_dir}")
if not val_dir.exists():
    raise FileNotFoundError(f"Не найден val split: {val_dir}")

train_sequences = sorted(path.name for path in train_dir.iterdir() if path.is_dir())
val_sequences = sorted(path.name for path in val_dir.iterdir() if path.is_dir())

print("train sequences:", len(train_sequences), train_sequences[:5])
print("val sequences:  ", len(val_sequences), val_sequences[:5])

## 4. Функции преобразования разметки

SportsMOT хранит ground truth в MOTChallenge-подобном формате. Для YOLO каждая рамка переводится в нормализованный формат `class x_center y_center width height`.

In [ ]:
def parse_seqinfo(seqinfo_path: Path) -> dict:
    cfg = configparser.ConfigParser()
    cfg.read(seqinfo_path)
    seq = cfg["Sequence"]
    return {
        "name": seq.get("name"),
        "imDir": seq.get("imDir", "img1"),
        "frameRate": seq.getint("frameRate", fallback=25),
        "seqLength": seq.getint("seqLength"),
        "imWidth": seq.getint("imWidth"),
        "imHeight": seq.getint("imHeight"),
        "imExt": seq.get("imExt", ".jpg"),
    }


def mot_line_to_yolo(parts: list[str], img_w: int, img_h: int) -> tuple[int, str] | None:
    if len(parts) < 6:
        return None

    frame_id = int(float(parts[0]))
    x = float(parts[2])
    y = float(parts[3])
    w = float(parts[4])
    h = float(parts[5])

    mark = int(float(parts[6])) if len(parts) > 6 else 1
    obj_class = int(float(parts[7])) if len(parts) > 7 else 1

    # В SportsMOT для обучения детектора используются только валидные игроки.
    if mark <= 0 or obj_class != 1 or w <= 1 or h <= 1:
        return None

    x1 = max(0.0, min(x, img_w - 1))
    y1 = max(0.0, min(y, img_h - 1))
    x2 = max(0.0, min(x + w, img_w))
    y2 = max(0.0, min(y + h, img_h))

    box_w = x2 - x1
    box_h = y2 - y1
    if box_w <= 1 or box_h <= 1:
        return None

    xc = x1 + box_w / 2
    yc = y1 + box_h / 2
    yolo_line = f"{CLASS_ID} {xc / img_w:.6f} {yc / img_h:.6f} {box_w / img_w:.6f} {box_h / img_h:.6f}"
    return frame_id, yolo_line


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def link_or_copy(src: Path, dst: Path, mode: str = "copy") -> None:
    ensure_parent(dst)
    if dst.exists():
        return
    if mode == "symlink":
        try:
            os.symlink(src, dst)
            return
        except OSError:
            pass
    shutil.copy2(src, dst)

In [ ]:
def convert_split(mot_split_dir: Path, out_root: Path, split_name: str, image_mode: str = "copy") -> dict[str, int]:
    out_images = out_root / "images" / split_name
    out_labels = out_root / "labels" / split_name
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    seq_dirs = sorted(path for path in mot_split_dir.iterdir() if path.is_dir())
    print(f"Converting split={split_name!r}, sequences={len(seq_dirs)}")

    stats = {"images": 0, "label_files": 0, "boxes": 0}

    for seq_dir in tqdm(seq_dirs, desc=split_name):
        seqinfo = parse_seqinfo(seq_dir / "seqinfo.ini")
        img_w, img_h = seqinfo["imWidth"], seqinfo["imHeight"]
        img_dir = seq_dir / seqinfo["imDir"]
        gt_path = seq_dir / "gt" / "gt.txt"

        labels_by_frame: dict[int, list[str]] = defaultdict(list)
        if gt_path.exists():
            for raw in gt_path.read_text(encoding="utf-8").splitlines():
                parts = [item.strip() for item in raw.split(",")]
                yolo_obj = mot_line_to_yolo(parts, img_w, img_h)
                if yolo_obj is None:
                    continue
                frame_id, yolo_line = yolo_obj
                labels_by_frame[frame_id].append(yolo_line)

        image_files = sorted(img_dir.glob("*"))
        for img_path in image_files:
            if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
                continue

            frame_id = int(img_path.stem)
            new_stem = f"{seq_dir.name}_{img_path.stem}"
            out_img_path = out_images / f"{new_stem}{img_path.suffix.lower()}"
            out_lbl_path = out_labels / f"{new_stem}.txt"

            link_or_copy(img_path, out_img_path, mode=image_mode)
            ensure_parent(out_lbl_path)
            out_lbl_path.write_text("\n".join(labels_by_frame.get(frame_id, [])), encoding="utf-8")

            stats["images"] += 1
            stats["label_files"] += 1
            stats["boxes"] += len(labels_by_frame.get(frame_id, []))

    print(f"[{split_name}] images={stats['images']}, label_files={stats['label_files']}, boxes={stats['boxes']}")
    return stats

## 5. Конвертация train/val split

In [ ]:
YOLO_ROOT.mkdir(parents=True, exist_ok=True)

train_stats = convert_split(train_dir, YOLO_ROOT, "train", image_mode=IMAGE_MODE)
val_stats = convert_split(val_dir, YOLO_ROOT, "val", image_mode=IMAGE_MODE)

print("train:", train_stats)
print("val:  ", val_stats)

## 6. Создание `sportsmot.yaml`

In [ ]:
data_yaml = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {CLASS_ID: CLASS_NAME},
}

yaml_path = YOLO_ROOT / "sportsmot.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(yaml_path)
print(yaml_path.read_text(encoding="utf-8"))

## 7. Контроль подготовленных файлов

In [ ]:
for path in [
    YOLO_ROOT / "images" / "train",
    YOLO_ROOT / "labels" / "train",
    YOLO_ROOT / "images" / "val",
    YOLO_ROOT / "labels" / "val",
    YOLO_ROOT / "sportsmot.yaml",
]:
    print(path, "->", path.exists())

print("train images:", len(list((YOLO_ROOT / "images" / "train").glob("*"))))
print("val images:  ", len(list((YOLO_ROOT / "images" / "val").glob("*"))))

## 8. Дообучение YOLO11

Результаты сохраняются в `YOLO_ROOT / "runs" / RUN_NAME`. Для получения трех детекторов нужно повторить запуск с `MODEL_NAME = "yolo11n.pt"`, `"yolo11s.pt"` и `"yolo11m.pt"`.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=str(YOLO_ROOT / "runs"),
    name=RUN_NAME,
    pretrained=True,
    exist_ok=True,
    cache=False,
    close_mosaic=10,
    seed=SEED,
    plots=True,
)

## 9. Продолжение обучения, если запуск был прерван

Этот блок по умолчанию выключен. Включите `RESUME_TRAINING = True`, если нужно продолжить обучение из `last.pt`.

In [ ]:
RESUME_TRAINING = False
LAST = YOLO_ROOT / "runs" / RUN_NAME / "weights" / "last.pt"

if RESUME_TRAINING:
    if not LAST.exists():
        raise FileNotFoundError(f"Не найден checkpoint: {LAST}")
    model = YOLO(str(LAST))
    results = model.train(resume=True)
else:
    print("Продолжение обучения выключено.")